# Notion-Zotero Analysis

Full pipeline: load canonical bundles → build summary tables → clean → explore.

**Data sources** (pick one):
-  — bundles pulled via 
-  — bundles pulled via 
-  — bundles parsed from local fixture files

In [ ]:
from pathlib import Path
import pandas as pd

from notion_zotero.analysis import (
    load_canonical_records,
    build_summary_dataframes,
    clean_table,
    run_analysis,
)

# Load from live pulled data
pulled_dir = Path("data/pulled/notion/learning_analytics_review")
if not pulled_dir.exists() or not any(pulled_dir.glob("*.canonical.json")):
    raise FileNotFoundError(
        f"No pulled data found in {pulled_dir}. "
        "Run: notion-zotero pull-notion"
    )

bundles = load_canonical_records(str(pulled_dir))
print(f"Loaded {len(bundles)} bundles from {pulled_dir}")

FileNotFoundError: No pulled data found in data\pulled\learning_analytics_review. Run: notion-zotero pull-notion

In [ ]:
# --- Cell 2: Inspect loaded bundles ---
print(f"Loaded {len(bundles)} canonical bundles")

if bundles:
    sample = bundles[0]
    refs = sample.get("references", [])
    if refs:
        r = refs[0]
        print(f"Sample: {r.get("title", "(no title)")} ({r.get("year", "?")})")

In [ ]:
# --- Cell 3: Build summary tables (task labels discovered from data) ---
def my_label_fn(task_name):
    mapping = {"prediction": "PRED", "description": "DESC",
               "knowledge tracing": "KT", "recommendation": "REC"}
    return mapping.get(task_name.lower(), task_name)

dfs = build_summary_dataframes(bundles, task_label_fn=my_label_fn)
for name, df in dfs.items():
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")
dfs.get("Reading List")

In [ ]:
# --- Cell 4: Clean tables (caller-provided fix dicts) ---
MY_TYPO_FIXES = {r"Fiiltering": "Filtering", r"Exerrcise": "Exercise"}
MY_VALUE_MAP  = {"n/a": "Not Applicable", "none": "Not Applicable", "na": "Not Applicable"}
SEARCH_COLS   = ["Search Strategy"]

cleaned_dfs = {}
for name, df in dfs.items():
    cleaned, log = clean_table(df, typo_fixes=MY_TYPO_FIXES, value_map=MY_VALUE_MAP, search_strategy_columns=SEARCH_COLS)
    cleaned_dfs[name] = cleaned
    print(f"  {name}: {log["text_updates"]} cell(s) normalised")

In [ ]:
# --- Cell 5: Shortcut — full pipeline ---
raw_dfs, clean_dfs, norm_log = run_analysis(
    pulled_dir,
    task_label_fn=my_label_fn,
    typo_fixes=MY_TYPO_FIXES,
    value_map=MY_VALUE_MAP,
    search_strategy_columns=SEARCH_COLS,
)
pd.DataFrame(norm_log).T

In [ ]:
# --- Cell 6: Explore ---
rl = clean_dfs.get("Reading List", pd.DataFrame())
if not rl.empty and "year" in rl.columns:
    display(rl["year"].value_counts().sort_index())
if not rl.empty and "journal" in rl.columns:
    display(rl["journal"].dropna().value_counts().head(10))
for name, df in clean_dfs.items():
    if name != "Reading List" and not df.empty:
        print(f"
{name} ({len(df)} rows):")
        display(df.head(5))